In [1]:
"""
Environment setup for MMDetection + DINO on Kaggle (T4 GPU, Python 3.12).

Run this ONCE per fresh Kaggle session, BEFORE running the training script.
All fixes below were needed because Kaggle's default image ships a very
recent torch/setuptools that conflicts with the mmdet/mmcv/mmengine stack:

  1. torch 2.10+cu128 (Kaggle default) has no matching prebuilt mmcv wheel
     -> pin torch to 2.4.1+cu121, which mmcv DOES have wheels for.
  2. `openmim`/`mim install` pulls in `openxlab`, which hard-pins
     setuptools~=60.2.0 -> that old setuptools breaks on Python 3.12
     (pkgutil.ImpImporter was removed). So we NEVER use `mim install`;
     we install mmengine/mmcv with plain pip instead.
  3. The very newest setuptools (80+) removed legacy `setup.py develop`,
     which mmdetection's install still relies on -> pin setuptools to a
     middle range (>=68,<80) that supports both Python 3.12 and legacy installs.
  4. mmdetection 3.3.0 has an off-by-one bug in its own version check:
     it rejects mmcv==2.2.0 even though 2.2.0 is the correct/required
     version for torch>=2.2 -> patch one line in mmdet/__init__.py
     (< to <=) after cloning. This is a known upstream bug, not a hack
     specific to your setup.

If anything fails partway through, do NOT skip steps -- rerun from where
it stopped, in order. Each step depends on the one before it.
"""

import subprocess
import sys

# ============================================================
# STEP 1. Pin torch to a version mmcv has prebuilt wheels for
# ============================================================
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch==2.4.1", "torchvision==0.19.1",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)

# ============================================================
# STEP 2. Pin setuptools to a version that:
#   - works on Python 3.12 (rules out <68)
#   - still supports legacy `setup.py develop` (rules out >=80)
# ============================================================
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "setuptools>=68,<80", "packaging>=24",
], check=True)

# ============================================================
# STEP 3. Install mmengine directly via pip -- NEVER use `mim install`
# (mim pulls in openxlab, which re-downgrades setuptools to 60.2.0
#  and reintroduces the pkgutil.ImpImporter crash on Python 3.12)
# ============================================================
subprocess.run([
    sys.executable, "-m", "pip", "install", "mmengine==0.10.4",
], check=True)

# ============================================================
# STEP 4. Install mmcv as a prebuilt wheel matching torch2.4 / cu121 / cp312
# ============================================================
subprocess.run([
    sys.executable, "-m", "pip", "install", "mmcv==2.2.0",
    "-f", "https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html",
], check=True)

# ============================================================
# STEP 5. Clone mmdetection (remove any partial/previous clone first)
# ============================================================
subprocess.run(["rm", "-rf", "mmdetection"], check=True)
subprocess.run([
    "git", "clone", "https://github.com/open-mmlab/mmdetection.git",
], check=True)

# ============================================================
# STEP 6. Install mmdetection in editable mode
# ============================================================
subprocess.run([
    sys.executable, "-m", "pip", "install", "-e", "mmdetection",
    "--no-build-isolation",
], check=True)

# ============================================================
# STEP 7. Patch mmdet's buggy mmcv-version check (< -> <=)
# mmdet 3.3.0 rejects mmcv==2.2.0 due to a strict "<" comparison against
# its own mmcv_maximum_version="2.2.0" -- this is an upstream off-by-one
# bug (see open-mmlab/mmdetection issue #12312), not specific to this env.
# ============================================================
init_path = "mmdetection/mmdet/__init__.py"
with open(init_path, "r") as f:
    content = f.read()

old = "mmcv_version < digit_version(mmcv_maximum_version)"
new = "mmcv_version <= digit_version(mmcv_maximum_version)"
if old in content:
    content = content.replace(old, new)
    with open(init_path, "w") as f:
        f.write(content)
    print(f"Patched {init_path}: relaxed mmcv_maximum_version check (< -> <=)")
else:
    print(f"[WARN] Expected patch target not found in {init_path} -- "
          "mmdetection's source may have changed; check manually if "
          "training still fails with an MMCV version AssertionError.")

# ============================================================
# STEP 8. Verify everything imports and CUDA is visible
# ============================================================
print("\n" + "=" * 70)
print("Verifying installed versions...")
print("=" * 70)
result = subprocess.run([
    sys.executable, "-c",
    "import torch, mmcv, mmengine, mmdet; "
    "print('torch:', torch.__version__, torch.version.cuda); "
    "print('mmcv:', mmcv.__version__); "
    "print('mmengine:', mmengine.__version__); "
    "print('mmdet:', mmdet.__version__); "
    "print('CUDA available:', torch.cuda.is_available())"
])
if result.returncode != 0:
    print("\n[ERROR] Verification failed -- do not proceed to training "
          "until the import above succeeds. Check the traceback for "
          "which package is missing or broken.")
    sys.exit(1)

print("\nEnvironment setup complete. You can now run the training script.")

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 103.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 201.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 109.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 83.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 84.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 187.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 141.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 83.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.1+cu121 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.7/451.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 14.5 MB/s eta 0:00:00
Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 14.2 MB/s eta 0:00:00


Cloning into 'mmdetection'...


Obtaining file:///kaggle/working/mmdetection
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for mmdet
Patched mmdetection/mmdet/__init__.py: relaxed mmcv_maximum_version check (< -> <=)

Verifying installed versions...
torch: 2.4.1+cu121 12.1
mmcv: 2.2.0
mmengine: 0.10.4
mmdet: 3.3.0
CUDA available: True

Environment setup complete. You can now run the training script.


In [2]:
"""
Train DINO (DETR with Improved deNoising Anchor boxes) on a custom
COCO-format detection dataset, using MMDetection.

Classes (nc=7):
    ['Bus', 'Jeepney', 'Motorcycle', 'Tricycle', 'Van', 'cars', 'truck']

HOW TO USE
----------
1. Run the install cell once (top of this file, under STEP 0) -- or use
   setup_mmdet_env.py.
2. Edit the paths in STEP 1 to point at your dataset.
3. If this is your FIRST run: leave RESUME_FROM = None below.
4. Run this script:  python train_dino_mmdet.py
5. After training finishes (or gets cut off by a session time limit),
   the training checkpoint is saved every epoch in WORK_DIR
   (default: ./work_dirs/dino_custom/).
6. If your session ends before reaching MAX_EPOCHS (e.g. Kaggle's 12h
   limit), on Kaggle: "Save Version" so /kaggle/working persists as a
   Dataset, then in your next session add that dataset as input and set
   RESUME_FROM below to the checkpoint's path (e.g.
   "/kaggle/input/<your-output-dataset>/work_dirs/dino_custom/best_coco_bbox_mAP_epoch_5.pth"),
   then rerun this same script. Training will pick up from that epoch
   and continue toward MAX_EPOCHS.
7. Whenever training completes (reaches MAX_EPOCHS), the script
   automatically runs test-set evaluation and prints COCO metrics
   (mAP, AP50, etc.) using the best checkpoint.

Expected dataset layout (Roboflow-style COCO export -- images and the
annotation file live together in the same folder):

    /content/dataset/
        train/
            <images>
            _annotations.coco.json
        valid/
            <images>
            _annotations.coco.json
        test/
            <images>
            _annotations.coco.json

If your COCO JSON's "categories" don't match the class list/order below,
this script will warn you loudly before training starts -- fix that first,
since a silent id/name mismatch is the most common cause of a model that
trains "fine" but never actually learns your classes correctly.
"""

import glob
import json
import os
import subprocess
import sys

# ============================================================
# STEP 0. INSTALL DEPENDENCIES (run once) -- or use setup_mmdet_env.py
# ============================================================
MMDET_ROOT = "mmdetection"  # path to the cloned mmdetection repo

# ============================================================
# STEP 1. DATASET PATHS -- EDIT THESE
# ============================================================

DATA_ROOT = "/kaggle/input/datasets/nomanhossain4/vehicles-green/Vehicles"

TRAIN_IMG_DIR = "train/"
TRAIN_ANN_FILE = "train/_annotations.coco.json"

VAL_IMG_DIR = "valid/"
VAL_ANN_FILE = "valid/_annotations.coco.json"

TEST_IMG_DIR = "test/"
TEST_ANN_FILE = "test/_annotations.coco.json"

# ============================================================
# STEP 2. CLASSES
# ============================================================

CLASS_NAMES = ("Bus", "Jeepney", "Motorcycle", "Tricycle", "Van", "cars", "truck")
NUM_CLASSES = len(CLASS_NAMES)  # 7

# ============================================================
# STEP 3. TRAINING HYPERPARAMETERS
# ============================================================

MAX_EPOCHS = 5          # keep this fixed at your real target across every
                         # run/session -- it drives the LR schedule
                         # (MultiStepLR milestone at epoch 11). Don't lower
                         # it to "test" something; use RESUME_FROM instead
                         # to pick up where a previous run left off.
BATCH_SIZE = 2
NUM_WORKERS = 4
BASE_LR = 0.0001
WORK_DIR = "./work_dirs/dino_custom"

PRETRAINED_CHECKPOINT = (
    "https://download.openmmlab.com/mmdetection/v3.0/dino/"
    "dino-4scale_r50_8xb2-12e_coco/"
    "dino-4scale_r50_8xb2-12e_coco_20221202_182705-55b2bba2.pth"
)

# ============================================================
# STEP 3b. RESUME FROM A PREVIOUS CHECKPOINT (if any)
# ============================================================
# Leave as None for a brand-new training run. If a previous run got cut
# off (e.g. Kaggle's session limit) and you saved that checkpoint as a
# Dataset, set this to its path -- training will continue from that
# epoch instead of starting over, and still train toward MAX_EPOCHS.
RESUME_FROM = None
# Example:
# RESUME_FROM = "/kaggle/input/my-dino-checkpoint-v1/work_dirs/dino_custom/best_coco_bbox_mAP_epoch_5.pth"


def sanity_check_coco_categories(ann_path, expected_names):
    """Warn (loudly) if the COCO json categories don't match expected_names/order."""
    if not os.path.exists(ann_path):
        print(f"[WARN] Annotation file not found yet: {ann_path}")
        return
    with open(ann_path, "r") as f:
        coco = json.load(f)
    cats = sorted(coco.get("categories", []), key=lambda c: c["id"])
    found_names = [c["name"] for c in cats]
    found_ids = [c["id"] for c in cats]

    print(f"Categories found in {ann_path}:")
    for cid, name in zip(found_ids, found_names):
        print(f"   id={cid:>2}  name={name}")

    if len(found_names) != len(expected_names):
        print(
            f"[WARN] Count mismatch: JSON has {len(found_names)} categories, "
            f"but nc={len(expected_names)} was given. Fix this before training."
        )
    elif list(found_names) != list(expected_names):
        print(
            "[WARN] Category NAMES/ORDER in the JSON do not match your yaml "
            f"list.\n  yaml order : {list(expected_names)}\n"
            f"  json order : {found_names}\n"
            "MMDetection's CocoDataset maps category IDs to indices based on "
            "the order in `metainfo['classes']` you pass in -- if that order "
            "doesn't line up with how the JSON's category ids were assigned, "
            "your labels will be silently wrong. Recommended fix: set "
            "CLASS_NAMES below to match the JSON's category order exactly."
        )
    else:
        print("Category names/order match your yaml list. Good to go.")


def build_config_text():
    """Return the text of a custom MMDetection config file for DINO."""
    metainfo = {
        "classes": CLASS_NAMES,
        "palette": [
            (220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 60, 100),
            (0, 80, 100), (0, 0, 230), (106, 0, 228),
        ][:NUM_CLASSES],
    }

    config_text = f'''
_base_ = [
    "{os.path.abspath(MMDET_ROOT)}/configs/dino/dino-4scale_r50_8xb2-12e_coco.py"
]

data_root = "{DATA_ROOT}/"

metainfo = {metainfo!r}

num_classes = {NUM_CLASSES}

# --- override the detection head's number of classes ---
model = dict(
    bbox_head=dict(num_classes=num_classes),
)

# --- train dataloader ---
train_dataloader = dict(
    batch_size={BATCH_SIZE},
    num_workers={NUM_WORKERS},
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file="{TRAIN_ANN_FILE}",
        data_prefix=dict(img="{TRAIN_IMG_DIR}"),
    ),
)

# --- val dataloader ---
val_dataloader = dict(
    batch_size=1,
    num_workers={NUM_WORKERS},
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file="{VAL_ANN_FILE}",
        data_prefix=dict(img="{VAL_IMG_DIR}"),
    ),
)

# --- test dataloader (held-out test split, separate from val) ---
test_dataloader = dict(
    batch_size=1,
    num_workers={NUM_WORKERS},
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file="{TEST_ANN_FILE}",
        data_prefix=dict(img="{TEST_IMG_DIR}"),
    ),
)

# --- evaluators must point at the matching annotation file for each split ---
val_evaluator = dict(
    ann_file=data_root + "{VAL_ANN_FILE}",
)
test_evaluator = dict(
    ann_file=data_root + "{TEST_ANN_FILE}",
)

# --- training schedule ---
max_epochs = {MAX_EPOCHS}
train_cfg = dict(max_epochs=max_epochs, val_interval=1)

# --- optimizer / LR ---
optim_wrapper = dict(
    optimizer=dict(lr={BASE_LR}),
)

# --- start from the official COCO-pretrained DINO checkpoint ---
load_from = "{PRETRAINED_CHECKPOINT}"

# --- checkpoint / logging ---
default_hooks = dict(
    checkpoint=dict(type="CheckpointHook", interval=1, max_keep_ckpts=3, save_best="auto"),
    logger=dict(type="LoggerHook", interval=50),
)

work_dir = "{WORK_DIR}"
'''
    return config_text


def find_best_checkpoint(work_dir):
    """
    Locate the checkpoint to evaluate with, preferring the 'best' one saved
    by save_best='auto' (e.g. best_coco_bbox_mAP_epoch_9.pth), falling back
    to the final epoch checkpoint, then to latest.pth.
    """
    best_ckpts = sorted(glob.glob(os.path.join(work_dir, "best_*.pth")))
    if best_ckpts:
        return best_ckpts[-1]

    epoch_ckpts = glob.glob(os.path.join(work_dir, "epoch_*.pth"))
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))
        return epoch_ckpts[-1]

    latest = os.path.join(work_dir, "latest.pth")
    if os.path.exists(latest):
        return latest

    return None


def run_test(config_path, work_dir):
    """
    Run MMDetection's tools/test.py against the TEST split defined in the
    config (test_dataloader / test_evaluator), using the best checkpoint
    from training. This prints COCO-style metrics: mAP, AP50, AP75,
    AP across small/medium/large objects, etc.
    """
    checkpoint_path = find_best_checkpoint(work_dir)
    if checkpoint_path is None:
        print(
            f"[ERROR] No checkpoint found in {work_dir} -- training may have "
            "failed or WORK_DIR is wrong. Skipping test evaluation."
        )
        return

    print(f"\nEvaluating checkpoint: {checkpoint_path}")

    test_script = os.path.join(MMDET_ROOT, "tools", "test.py")
    results_file = os.path.join(work_dir, "test_results.pkl")
    work_dir_test = os.path.join(work_dir, "test_eval")
    os.makedirs(work_dir_test, exist_ok=True)

    cmd = [
        sys.executable, test_script,
        config_path,
        checkpoint_path,
        "--out", results_file,
        "--work-dir", work_dir_test,
    ]
    print("\nLaunching test evaluation with command:")
    print("  " + " ".join(cmd))
    print("=" * 70 + "\n")

    subprocess.run(cmd, check=True)

    print("\n" + "=" * 70)
    print(f"Test evaluation complete.")
    print(f"  Raw predictions saved to: {results_file}")
    print(f"  Full logs saved to:       {work_dir_test}")
    print("  COCO metrics (mAP, AP50, AP75, AP_small/medium/large, AR...) are")
    print("  printed above by tools/test.py and also saved in the log file.")
    print("=" * 70)


def main():
    print("=" * 70)
    print("DINO (MMDetection) training setup")
    print("=" * 70)

    if not os.path.isdir(MMDET_ROOT):
        print(
            f"[ERROR] '{MMDET_ROOT}' not found. Run the install commands in "
            "STEP 0 first (clone + pip install -e mmdetection)."
        )
        sys.exit(1)

    # Sanity-check train/val/test annotation files against your class list
    sanity_check_coco_categories(os.path.join(DATA_ROOT, TRAIN_ANN_FILE), CLASS_NAMES)
    sanity_check_coco_categories(os.path.join(DATA_ROOT, VAL_ANN_FILE), CLASS_NAMES)
    sanity_check_coco_categories(os.path.join(DATA_ROOT, TEST_ANN_FILE), CLASS_NAMES)

    os.makedirs(WORK_DIR, exist_ok=True)
    config_path = os.path.join(WORK_DIR, "dino_custom_config.py")
    with open(config_path, "w") as f:
        f.write(build_config_text())
    print(f"\nWrote custom config to: {config_path}")

    train_script = os.path.join(MMDET_ROOT, "tools", "train.py")
    cmd = [sys.executable, train_script, config_path]

    # --- resume support: pass --resume only if a checkpoint path was given ---
    if RESUME_FROM:
        if not os.path.exists(RESUME_FROM):
            print(f"[ERROR] RESUME_FROM is set but the path doesn't exist: {RESUME_FROM}")
            print("Double-check the checkpoint filename in your input dataset "
                  "and fix RESUME_FROM before rerunning.")
            sys.exit(1)
        cmd += ["--resume", RESUME_FROM]
        print(f"Resuming training from checkpoint: {RESUME_FROM}")
    else:
        print("Starting a fresh training run (RESUME_FROM is None).")

    print("\nLaunching training with command:")
    print("  " + " ".join(cmd))
    print("=" * 70 + "\n")

    subprocess.run(cmd, check=True)

    # ------------------------------------------------------------
    # STEP 4. TEST / EVALUATION -- always runs after training finishes
    # ------------------------------------------------------------
    run_test(config_path, WORK_DIR)


if __name__ == "__main__":
    main()

DINO (MMDetection) training setup
Categories found in /kaggle/input/datasets/nomanhossain4/vehicles-green/Vehicles/train/_annotations.coco.json:
   id= 0  name=Cars,Truck,Bike,Motorcyle
   id= 1  name=Bus
   id= 2  name=Jeepney
   id= 3  name=Motorcycle
   id= 4  name=Tricycle
   id= 5  name=Van
   id= 6  name=cars
   id= 7  name=truck
[WARN] Count mismatch: JSON has 8 categories, but nc=7 was given. Fix this before training.
Categories found in /kaggle/input/datasets/nomanhossain4/vehicles-green/Vehicles/valid/_annotations.coco.json:
   id= 0  name=Cars,Truck,Bike,Motorcyle
   id= 1  name=Bus
   id= 2  name=Jeepney
   id= 3  name=Motorcycle
   id= 4  name=Tricycle
   id= 5  name=Van
   id= 6  name=cars
   id= 7  name=truck
[WARN] Count mismatch: JSON has 8 categories, but nc=7 was given. Fix this before training.
Categories found in /kaggle/input/datasets/nomanhossain4/vehicles-green/Vehicles/test/_annotations.coco.json:
   id= 0  name=Cars,Truck,Bike,Motorcyle
   id= 1  name=Bus
   i

/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


09/13 14:40:55 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 870176782
    GPU 0,1: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.4.2 (Git Hash 1137e04ec0b5251ca2b4400a4fd3c667ce843d67)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,cod

/kaggle/working/mmdetection/mmdet/models/backbones/trident_resnet.py:244: SyntaxWarning: invalid escape sequence '\ '
  \ stage3(b2) /
/kaggle/working/mmdetection/mmdet/models/dense_heads/free_anchor_retina_head.py:290: SyntaxWarning: invalid escape sequence '\i'
  :math:`FL((1 - P_{a_{j} \in A_{+}}) * (1 - P_{j}^{bg}))`.


09/13 14:40:58 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
09/13 14:40:58 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                

...

09/13 22:51:33 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 384098212
    GPU 0,1: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.4.2 (Git Hash 1137e04ec0b5251ca2b4400a4fd3c667ce843d67)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,cod

/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_l

Loads checkpoint by local backend from path: ./work_dirs/dino_custom/best_coco_bbox_mAP_epoch_5.pth
09/13 22:51:39 - mmengine - INFO - Load checkpoint from ./work_dirs/dino_custom/best_coco_bbox_mAP_epoch_5.pth


/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


09/13 22:51:47 - mmengine - INFO - Epoch(test) [ 50/566]    eta: 0:01:27  time: 0.1700  data_time: 0.0056  memory: 384  
09/13 22:51:55 - mmengine - INFO - Epoch(test) [100/566]    eta: 0:01:15  time: 0.1547  data_time: 0.0027  memory: 384  
09/13 22:52:03 - mmengine - INFO - Epoch(test) [150/566]    eta: 0:01:07  time: 0.1589  data_time: 0.0026  memory: 384  
09/13 22:52:11 - mmengine - INFO - Epoch(test) [200/566]    eta: 0:00:59  time: 0.1631  data_time: 0.0026  memory: 384  
09/13 22:52:19 - mmengine - INFO - Epoch(test) [250/566]    eta: 0:00:51  time: 0.1646  data_time: 0.0028  memory: 384  
09/13 22:52:27 - mmengine - INFO - Epoch(test) [300/566]    eta: 0:00:43  time: 0.1600  data_time: 0.0025  memory: 384  
09/13 22:52:35 - mmengine - INFO - Epoch(test) [350/566]    eta: 0:00:34  time: 0.1561  data_time: 0.0026  memory: 384  
09/13 22:52:43 - mmengine - INFO - Epoch(test) [400/566]    eta: 0:00:26  time: 0.1541  data_time: 0.0027  memory: 384  
09/13 22:52:51 - mmengine - INFO